# Hamilton PREP: Concise demo — teaching needle, liquid transfer, plate movement

Single notebook demonstrating:
1. **Teaching needle** — Pick up the teaching needle, move above plate A1 at safe height, drop it.
2. **Dual-channel liquid handling** — Tip pickup, aspirate, and dispense (tip + volume tracking).
3. **8MPH (`head8`) liquid handling** — Full-column tip pickup, aspirate, and dispense when `head8_installed`.
4. **Plate movement** — CoRe gripper: pick plate from deck[4], drop at deck[2].

**Deck layout:** 1× 50 µL NTR tips at deck[3], 1× plate at deck[4] (moved to deck[2]). Visualizer is rooted on the deck: tip-spot occupancy, well fills, and plate assignment update live when tracking is enabled. Pipette mount state lives on each channel's mounting shaft (`prep.pipettes.get_mounted_tip(ch)`) and on `prep.head8.get_mounted_tips()` / `prep.head8.shaft(i)` (printed below; not yet wired into the visualizer pipette panel).

Uses {func}`~pylabrobot.hamilton.prep.device.Prep` with `prep.pipettes`, `prep.head8`, and `prep.core_grippers`. Firmware tree / command-signature dumps live in the channel introspection notebooks.


## 1. Imports and config


In [ ]:
import logging
import sys
from asyncio import sleep

from pylabrobot.hamilton.prep import Prep
from pylabrobot.resources import Coordinate, set_tip_tracking, set_volume_tracking
from pylabrobot.resources.corning.axygen.plates import cor_axy_96_wellplate_500uL_Ub
from pylabrobot.resources.hamilton import PrepDeck, hamilton_96_tiprack_50uL_NTR
from pylabrobot.visualizer import Visualizer

logging.getLogger("pylabrobot").setLevel(logging.INFO)
logging.getLogger("pylabrobot").handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
logging.getLogger("pylabrobot").addHandler(handler)

# Opt-in labware tracking (drives TipSpot / well updates in Visualizer(deck)).
set_tip_tracking(True)
set_volume_tracking(True)

HOST = "192.168.100.102" # "127.0.0.1" For port forwarded connection if set up
PORT = 2000
SAFE_HEIGHT_MM_ABOVE_WELL = 20


## 2. Deck layout and visualizer


In [ ]:
# PrepDeck: spots 0–7 (column-major). With CoRe grippers mount for plate movement.
deck = PrepDeck(with_core_grippers=True)

tip_rack = deck[3] = hamilton_96_tiprack_50uL_NTR(name="ntr_50", with_tips=True)
plate = deck[4] = cor_axy_96_wellplate_500uL_Ub("plate")

visualizer = Visualizer(deck, open_browser=False)
await visualizer.setup()


## 3. Prep device and setup


In [ ]:
prep = Prep(deck=deck, host=HOST, port=PORT)
await prep.setup(smart=True, force_initialize=False)


In [ ]:
await prep.lights.turn_on()


## 4. Device snapshot


In [ ]:
cfg = prep.driver.configuration
print(
  f"channels={cfg.num_channels}, head8_installed={cfg.head8_installed}, "
  f"traverse_height={prep.pipettes.default_minimum_traverse_height}, "
  f"enclosure={cfg.has_enclosure}, safe_speeds={cfg.safe_speeds_enabled}"
)


## 5. Teaching needle: above plate A1 at safe height


In [ ]:
logging.getLogger("pylabrobot").setLevel(logging.DEBUG)

assert prep.pipettes is not None
teaching_needle = deck.teaching_needle_spot
assert teaching_needle is not None
if not teaching_needle.has_tip():
  teaching_needle.tracker.add_tip(teaching_needle.make_tip(), origin=teaching_needle, commit=True)

await prep.pipettes.pick_up_tips([teaching_needle], use_channels=[0])

a1 = plate.get_item("A1")
safe_pos = a1.get_absolute_location("c", "c", "b") + Coordinate(0, 0, SAFE_HEIGHT_MM_ABOVE_WELL)
await prep.pipettes.move_to_location(safe_pos, use_channels=[0])
await sleep(3)

await prep.pipettes.drop_tips([teaching_needle], use_channels=[0])

logging.getLogger("pylabrobot").setLevel(logging.INFO)


## 6. Tip pickup, aspirate, dispense (dual channel)

Tip and volume tracking are on: tip spots and well fills update in the visualizer; each channel's mounting shaft holds its tip (printed below).


In [ ]:
assert prep.pipettes is not None
tip_spots = tip_rack["A1:B1"]
channels = [0, 1]
src = plate["A1:B1"]
dst = plate["A7:B7"]
vols = [35.0, 25.0]

for well in src:
  well.tracker.set_volume(100.0)

def _status(label: str) -> None:
  print(label)
  print(f"  tip spots have tip: {[s.has_tip() for s in tip_spots]}")
  print(f"  mounted tips: {prep.pipettes.get_mounted_tips()}")
  print(f"  src volumes: {[w.tracker.get_used_volume() for w in src]}")
  print(f"  dst volumes: {[w.tracker.get_used_volume() for w in dst]}")
  tips = [prep.pipettes.get_mounted_tip(ch) for ch in channels]
  print(f"  tip volumes: {[t.tracker.get_used_volume() if t is not None else None for t in tips]}")

_status("before pick")
await prep.pipettes.pick_up_tips(tip_spots, use_channels=channels)
_status("after pick")

await prep.pipettes.aspirate(
  src,
  volumes=vols,
  use_channels=channels,
  liquid_heights=[3.0, 3.0],
  swap_speeds=[25.0, 25.0],
)
_status("after aspirate")

await prep.pipettes.dispense(
  dst,
  volumes=vols,
  use_channels=channels,
  liquid_heights=[3.0, 3.0],
  swap_speeds=[25.0, 25.0],
)
_status("after dispense")

await prep.pipettes.drop_tips(tip_spots, use_channels=channels)
_status("after drop")


## 7. Tip pickup, aspirate, dispense (8MPH / head8)

Skipped when `prep.head8` is None. Uses tip rack `A2:H2` and plate `A2:H2` → `A4:H4` so it does not collide with the dual-channel wells above. All 8 probes operate together.


In [ ]:
if prep.head8 is None:
  print("head8 not present (head8_installed=False); skipping 8MPH transfer")
else:
  tip_spots8 = tip_rack["A2:H2"]
  src8 = plate["A2:H2"]
  dst8 = plate["A4:H4"]
  vol8 = 15.0
  for well in src8:
    well.tracker.set_volume(100.0)

  def _status8(label: str) -> None:
    print(label)
    print(f"  tip spots have tip: {[s.has_tip() for s in tip_spots8]}")
    print(f"  mounted tips: {prep.head8.get_mounted_tips()}")
    print(f"  src volumes: {[w.tracker.get_used_volume() for w in src8]}")
    print(f"  dst volumes: {[w.tracker.get_used_volume() for w in dst8]}")
    tips = [prep.head8.get_mounted_tip(i) for i in range(8)]
    print(f"  tip volumes: {[t.tracker.get_used_volume() if t is not None else None for t in tips]}")

  _status8("before pick8")
  await prep.head8.pick_up_tips(tip_spots8)
  _status8("after pick8")

  await prep.head8.aspirate(containers=src8, volume=vol8, liquid_height=3.0)
  _status8("after aspirate")

  await prep.head8.dispense(containers=dst8, volume=vol8, liquid_height=3.0)
  _status8("after dispense")

  await prep.head8.drop_tips(tip_spots8)
  _status8("after drop8")


## 8. Plate movement with CoRe gripper (deck[4] → deck[2])

`drop_resource` reassigns the plate in the resource tree; the visualizer moves it to deck[2].


In [ ]:
async with prep.core_grippers.mounted() as grippers:
    await grippers.pick_up_resource(plate)
    await grippers.drop_resource(deck[2])
print(f"plate parent: {plate.parent.name if plate.parent is not None else None}")


## 9. Teardown


In [ ]:
await prep.driver.park()
#await prep.lights.disco()
await prep.stop()
await visualizer.stop()
